In [ ]:
#pip install h5py nibabel pandas glob2 regex
import os
import sys
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
from tqdm import tqdm
from itertools import product
import scipy
import torch
import torch.nn as nn
os.environ["CUDA_VISIBLE_DEVICES"]= '1' #, this way I would choose GPU 3 to do the work
import scipy.io as sio
sys.path.append(os.path.abspath(".."))
sys.path.append(os.path.abspath("../src"))

from walinet.inference.inference import * 
from walinet.inference.fid_inference import infer_fid
from walinet.training_data.lipid_removal import * 
from walinet.visualization.EvaluateModel import * 

# Run-Config
Scanner = "7T"

sub = "UCSF/Vol1"

add_water = False

if Scanner == "3T":
    Vol = "B0corrected_wo_LipidMask/PRISMA_Vienna/Vol01_WB/Res50x50"
    version = 'v_1.1'
    model_1 = '3T_Final_32_Channels'
    #model_2 = '3T_50x50_Unet_1.1_6Layer'

elif Scanner == "7T":
    Vol = f"B0corrected_wo_LipidMask/{sub}" #f"B0corrected_wo_LipidMask/{sub}" #"NoB0Correction/MS_{MS}"#"NoB0Correction/MS_180" #"B0corrected_wo_LipidMask/Vol5"
    version = 'v_1.0' # data version
    model_1 = '7T_Final' #'UNet_ablation_parameter_matched'
   # model_2 = 'UNet_ablation_parameter_matched'#"7T_Unet_1.1_6Layers"#'7T_Ynet_Final_proj_en_1.1'    #'Final_B0correction_In_Brain', "Vienna_5DataSets_1.1_7Layers"
     #max_abs' #projection_enery'

In [ ]:
dat = "/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/home/bstrasser/CooperationPartners/Gosia_Minnesota/Results/Measurement27/mrsi_x27_stability_CRT2_1/CombinedCSI.mat"

import h5py

with h5py.File(dat, "r") as f:
    data = f["csi"]["Data"][:]

# MATLAB-Achsenreihenfolge wiederherstellen
data = data.transpose()

# Falls komplexe Daten als real/imag gespeichert sind
if data.dtype.fields is not None:
    data = data["real"] + 1j * data["imag"]

print(data.shape)
print(data.dtype)

In [ ]:
%matplotlib widget

# Interactive viewer for the single dataset loaded above.
# The anatomical/background panel is the magnitude of the fourth FID point.
fid_point = min(3, data.shape[-1] - 1)
viewer_image = np.nan_to_num(
    np.abs(data[..., fid_point]), nan=0.0, posinf=0.0, neginf=0.0
)
viewer_spectra = np.real(
    np.fft.fftshift(np.fft.fft(data, axis=-1), axes=-1)
)

# Acquisition parameters used elsewhere in this notebook/project.
# Adjust these values here if a dataset has different scan parameters.
if Scanner == '7T':
    ppm = make_ppm_axis(
        vec_size=data.shape[-1],
        dwelltime=360000,
        larmor_freq=297222876,
        ppm_ref=4.7,
        ref_freq_hz=0.0,
        dwelltime_unit='ns',
    )
elif Scanner == '3T':
    ppm = make_ppm_axis(
        vec_size=data.shape[-1],
        dwelltime=1063800,
        larmor_freq=123231706,
        ppm_ref=4.7,
        ref_freq_hz=0.0,
        dwelltime_unit='ns',
    )
else:
    raise ValueError(f'No ppm-axis parameters configured for Scanner={Scanner!r}.')

# Begin at the strongest voxel in the fourth-FID-point magnitude map.
x_init, y_init, z_init = np.unravel_index(
    np.argmax(viewer_image), viewer_image.shape
)

interactive_spectra_viewer_multi(
    image_volume=viewer_image,
    spectrum_funcs=[make_spec_func(viewer_spectra)],
    labels=['Loaded dataset'],
    x_axis=ppm,
    x_label='ppm',
    ppm_min=0,
    ppm_max=7,
    x_init=int(x_init),
    y_init=int(y_init),
    z_init=int(z_init),
    overlay_first_two=False,
    ncols_spec=1,
    compact=True,
    legend=False,
    show_image_axes=True,
)
